# 07 — FactGraph V1 Query, Scenario, Run, Explain, and Portable Engines

This notebook is a runnable tour of the final FactGraph V1 closure. It uses the public `SDKStore.query(...)` execution entrypoint and a small amount of application-level rule and Policy authoring to demonstrate the new execution contract end to end.

> **Layering note.** `fg.query(...).bind(...).select(...).plan().run()` is the shipped SDK execution surface. `PolicyAll`, `PolicyAny`, `PolicyOccurrence`, `PolicyCompare`, `SemanticAddressSpace`, and `SemanticPortAddress` are currently advanced application/compiler authoring objects. They are shown here so the sealed Policy/Evidence contract is inspectable; they are **not** the intended business-user Policy authoring API. A future ergonomic Policy builder should create typed occurrence/port handles and keep these objects behind it.

It covers:

1. one Query shape for a resolved Rule and an immutable Policy;
2. `GoalPlanV1` result modes and expectations;
3. grounded, non-persistent `ScenarioSpecV1` What-if worlds;
4. detached replay and explicit positive-row Explain;
5. full Policy topology (`All`, `Any`, `Unify`, `Compare`, and field navigation);
6. real Native/Soufflé/ProbLog selected-row parity;
7. a constrained pre-engine relation provider; and
8. immutable candidate comparison.

The notebook deliberately calls out the semantic boundaries. It does **not** teach Action execution, Agent permissions, Package lookup, source-authority decisions, global closed-world negation, or proof equivalence across engines.

## How to run it

Launch Jupyter from the repository root (or `examples/`) with the `factpy` kernel. The setup cell adds `src/` to `sys.path`, so no package installation is needed for an in-repository checkout.

Every code cell asserts the contract it demonstrates. The portable-engine cell requires the same Native, Soufflé, and ProbLog installations used by the FactGraph test suite.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
_src_dir = _repo_root / 'src'
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from factgraph.application import (
    SemanticAddressSpace,
    build_resolved_rule,
    build_schema_index,
    field_predicate,
    manage_rule_occurrence,
)
from factgraph.application.protocol import (
    EvaluationQueryFieldNavigationV0,
    Policy,
    PolicyAll,
    PolicyAny,
    PolicyCompare,
    PolicyFieldNavigation,
    PolicyOccurrence,
    PolicyUnify,
    SemanticPortAddress,
    SemanticRulePort,
    entity_identity,
    field_endpoint,
)
from factgraph.core.protocol.digests import sha256_token
from factgraph.core.rules.where_ast import CmpAtom, Const, PredAtom, Var
from factgraph.sdk import (
    ContainsRowExpectationV1,
    CountEqExpectationV1,
    Entity,
    EntityRef,
    ExactLocalAbsenceExpectationV1,
    ExactLocalClosureTargetV1,
    ExistsExpectationV1,
    ExplainTargetV1,
    Field,
    FieldPath,
    GoalPlanRunV1,
    GoalRowExpectationV1,
    GoalValueV1,
    Identity,
    ProviderMaterializationV1,
    ProviderRelationRowV1,
    RelationProviderV1,
    FactGraph,
    ScenarioSetEffectiveValueV1,
    ScenarioSpecV1,
    ScenarioValueV1,
    ScenarioWithoutFieldV1,
    SetEqualsExpectationV1,
    portable_deterministic_profile_v1,
    provider_binding_slot_v1,
)


def address(occurrence: str, port: str) -> SemanticPortAddress:
    """A structured semantic address — never a dotted string path."""

    return SemanticPortAddress(occurrence, port)


## 1. A small schema, facts, and reusable Rules

The Rule authoring below is deliberately explicit: Rule ports are typed semantic endpoints. A Query later binds and selects those ports using `SemanticPortAddress`; it never guesses parameter positions from names.

The fixture uses the public `FactGraph.entities.create(...)` and `FactGraph.fields.set(...)` APIs; the V1 examples then use the SDK Query surface.

In [ ]:
class Person(Entity):
    employee_id: str = Identity()
    age: int = Field()
    score: int = Field()


def person_values_rule(graph: FactGraph):
    index = build_schema_index(graph.schema_ir)
    person, age, score = Var('$person'), Var('$age'), Var('$score')
    return build_resolved_rule(
        id='tutorial_person_values',
        version='1',
        when=(
            PredAtom('Person:exists', [person]),
            PredAtom('person:age', [person, age]),
            PredAtom('person:score', [person, score]),
        ),
        ports={
            'person': SemanticRulePort(person, entity_identity('Person')),
            'age': SemanticRulePort(age, field_endpoint('Person', 'age')),
            'score': SemanticRulePort(score, field_endpoint('Person', 'score')),
        },
        schema_index=index,
    )


def age_only_rule(graph: FactGraph):
    index = build_schema_index(graph.schema_ir)
    person, age = Var('$person'), Var('$age')
    return build_resolved_rule(
        id='tutorial_person_age',
        version='1',
        when=(
            PredAtom('Person:exists', [person]),
            PredAtom('person:age', [person, age]),
        ),
        ports={
            'person': SemanticRulePort(person, entity_identity('Person')),
            'age': SemanticRulePort(age, field_endpoint('Person', 'age')),
        },
        schema_index=index,
    )


def age_with_score_guard_rule(graph: FactGraph):
    index = build_schema_index(graph.schema_ir)
    person, age, score = Var('$person'), Var('$age'), Var('$score')
    return build_resolved_rule(
        id='tutorial_age_with_score_guard',
        version='1',
        when=(
            PredAtom('Person:exists', [person]),
            PredAtom('person:age', [person, age]),
            PredAtom('person:score', [person, score]),
        ),
        ports={
            'person': SemanticRulePort(person, entity_identity('Person')),
            'age': SemanticRulePort(age, field_endpoint('Person', 'age')),
        },
        schema_index=index,
    )


def threshold_rule(graph: FactGraph, *, rule_id: str, threshold: int):
    index = build_schema_index(graph.schema_ir)
    person, age = Var('$person'), Var('$age')
    return build_resolved_rule(
        id=rule_id,
        version='1',
        when=(
            PredAtom('Person:exists', [person]),
            PredAtom('person:age', [person, age]),
            CmpAtom('gt', age, Const(threshold)),
        ),
        ports={
            'person': SemanticRulePort(person, entity_identity('Person')),
            'age': SemanticRulePort(age, field_endpoint('Person', 'age')),
        },
        schema_index=index,
    )


def identity_rule(graph: FactGraph):
    index = build_schema_index(graph.schema_ir)
    person = Var('$person')
    return build_resolved_rule(
        id='tutorial_person_identity',
        version='1',
        when=(PredAtom('Person:exists', [person]),),
        ports={'person': SemanticRulePort(person, entity_identity('Person'))},
        schema_index=index,
    )


graph = FactGraph.create(schema_classes=[Person])
alice_ref = graph.entities.create(Person, employee_id='alice')
graph.fields.set(Person.age, alice_ref, 22)
graph.fields.set(Person.score, alice_ref, 9)
bob_ref = graph.entities.create(Person, employee_id='bob')
graph.fields.set(Person.age, bob_ref, 19)
graph.fields.set(Person.score, bob_ref, 7)
alice = EntityRef('Person', {'employee_id': 'alice'})
bob = EntityRef('Person', {'employee_id': 'bob'})

assert alice_ref.startswith('idref_v1:')
assert bob_ref.startswith('idref_v1:')


## 2. Rule and `GoalPlanV1`: one declared query contract

A V1 Query does not execute while it is being assembled. `.plan()` freezes the target, structured bindings, selections, expectation inventory, Scenario request, provider identity, and execution profile into a `GoalPlanV1` invocation. `.run()` produces a sealed `EvaluationRunV1`.

A single Rule is automatically lifted into a one-occurrence Policy internally; callers do not need a Policy registry or a policy-name string.

In [ ]:
basic = (
    graph.query(person_values_rule(graph))
    .bind(address('target', 'person'), alice)
    .select('age', address('target', 'age'))
    .plan()
    .run()
)
assert isinstance(basic, GoalPlanRunV1)

row = basic.run.effective.canonical_result.rows[0]
assert dict(row.values)['age'].value == 22
assert basic.run.plan.target.kind == 'rule'

{
    'query_digest': basic.run.plan.query_digest,
    'result': {alias: value.value for alias, value in row.values},
    'completeness': basic.run.effective.canonical_result.completeness,
}


## 3. Result modes, completeness, and expectations

Rows, `exists`, `count`, and set equality are distinct result concepts. Expectations retain their own status and completeness basis. In particular, a missing row becomes `not_satisfied` only after complete enumeration; it is never silently treated as a logical negative fact.

In [ ]:
age_19 = GoalRowExpectationV1((('age', GoalValueV1('int', 19)),))
age_22 = GoalRowExpectationV1((('age', GoalValueV1('int', 22)),))
expectation_run = (
    graph.query(age_only_rule(graph))
    .select('age', address('target', 'age'))
    .plan(
        expectations=(
            ContainsRowExpectationV1('has-19', age_19),
            ExistsExpectationV1('has-anyone', True),
            CountEqExpectationV1('two-people', 2),
            SetEqualsExpectationV1('exact-age-set', (age_19, age_22)),
        )
    )
    .run()
)
assert isinstance(expectation_run, GoalPlanRunV1)
outcomes = {
    item.expectation_id: item.status
    for item in expectation_run.run.effective.canonical_result.expectation_outcomes
}
assert outcomes == {
    'has-19': 'satisfied',
    'has-anyone': 'satisfied',
    'two-people': 'satisfied',
    'exact-age-set': 'satisfied',
}
outcomes


## 4. What-if is a sealed effective world, not a ledger write

`ScenarioSpecV1` carries grounded operations. FactGraph resolves conflicts and materializes a baseline and an effective world before evaluation. The source `SDKStore` is never changed. The resulting `ScenarioDiffV1` is an input/result comparison, not a causal explanation or a product verdict.

In [ ]:
replacement = ScenarioSpecV1(
    (
        ScenarioSetEffectiveValueV1(
            'alice-age-35',
            alice,
            FieldPath('Person', 'age'),
            ScenarioValueV1('int', 35),
            origin_refs=('tutorial:age-hypothesis',),
        ),
    )
)
what_if = (
    graph.query(age_only_rule(graph))
    .bind(address('target', 'person'), alice)
    .select('age', address('target', 'age'))
    .what_if(replacement)
    .plan()
    .run()
)
assert isinstance(what_if, GoalPlanRunV1)
baseline_age = dict(what_if.run.baseline.canonical_result.rows[0].values)['age'].value
effective_age = dict(what_if.run.effective.canonical_result.rows[0].values)['age'].value
assert (baseline_age, effective_age) == (22, 35)
assert what_if.scenario_diff is not None
assert what_if.scenario_diff.result_relation == 'different'
assert what_if.scenario_diff.causal_attribution == 'not_claimed'

{'baseline_age': baseline_age, 'effective_age': effective_age, 'diff_axes': what_if.scenario_diff.input_difference_axes}


### Exact local absence is not global negation

`ScenarioWithoutFieldV1` makes one exact Scenario target empty. Pairing it with `ExactLocalAbsenceExpectationV1` can prove that narrow closed target in the effective world. The baseline remains `underdetermined`; an empty result row alone never becomes a general `false` or a negative `EvidenceGraph`.

In [ ]:
index = build_schema_index(graph.schema_ir)
age_predicate_id = field_predicate(index, 'Person', 'age').pred_id
absence_scenario = ScenarioSpecV1(
    (ScenarioWithoutFieldV1('remove-alice-age', alice, FieldPath('Person', 'age')), )
)
absence_expectation = ExactLocalAbsenceExpectationV1(
    'alice-age-is-empty-in-this-scenario',
    ExactLocalClosureTargetV1('field', alice_ref, age_predicate_id),
)
absence_run = (
    graph.query(age_only_rule(graph))
    .bind(address('target', 'person'), alice)
    .select('age', address('target', 'age'))
    .what_if(absence_scenario)
    .plan(expectations=(absence_expectation,))
    .run()
)
assert isinstance(absence_run, GoalPlanRunV1)
assert absence_run.run.baseline.canonical_result.expectation_outcomes[0].status == 'underdetermined'
assert absence_run.run.effective.canonical_result.expectation_outcomes[0].status == 'satisfied'
assert absence_run.run.effective.canonical_result.rows == ()

summary = absence_run.run.effective.canonical_result.summary_anchor
assert summary is not None
summary_explain = absence_run.explain(
    ExplainTargetV1('effective', 'summary', summary.summary_anchor_digest)
)
assert summary_explain.evidence_graph is None
assert summary_explain.logical_conclusion == 'not_claimed'
{
    'baseline_expectation': absence_run.run.baseline.canonical_result.expectation_outcomes[0].status,
    'effective_expectation': absence_run.run.effective.canonical_result.expectation_outcomes[0].status,
    'summary_graph': summary_explain.evidence_graph,
}


## 5. A Policy is an authored topology, not an `AND`/`OR` string

This section deliberately uses the current **advanced authoring layer**, not a proposed business SDK. A direct `Policy` supplies its own semantic address space. The compiler preserves its authored nodes and lineage separately from lowered DNF execution branches. That is why a V1 Explain can project evidence back into the original `All` / `Any` / occurrence / `Unify` / `Compare` topology.

First, here is a simple direct Policy plus one-hop field navigation.

In [ ]:
identity = identity_rule(graph)
identity_space = SemanticAddressSpace((manage_rule_occurrence(identity, 'person'),))
identity_policy = Policy('tutorial_people', PolicyOccurrence('person'), version='1')
policy_navigation = (
    graph.query(identity_policy, address_space=identity_space)
    .bind(address('person', 'person'), alice)
    .select(
        'age',
        EvaluationQueryFieldNavigationV0(
            address('person', 'person'), FieldPath('Person', 'age')
        ),
    )
    .plan()
    .run()
)
assert isinstance(policy_navigation, GoalPlanRunV1)
assert policy_navigation.run.plan.target.kind == 'policy'
assert dict(policy_navigation.run.effective.canonical_result.rows[0].values)['age'].value == 22
'Policy and Rule share the same .bind(...).select(...).plan().run() path.'


### Detached Explain: graph plus Policy overlay

The next Policy has a holding `Any` arm and a failing arm whose later comparison is `NOT_REACHED`. The run seals a **restricted native Explain context**, not a prebuilt EvidenceGraph. When an explicit positive row is requested, Explain revalidates the context and deterministically recomputes Native logical evidence over the sealed captured relation.

This is what aligns EvidenceGraph with the newer Policy form: evidence stays an engine-level inner graph, while `policy_projection` maps it through the sealed authored Policy structure and lineage. For a portable run, this is still native inner evidence; proof parity across engines remains `not_claimed`.

In [ ]:
common = person_values_rule(graph)
eligible = threshold_rule(graph, rule_id='tutorial_eligible', threshold=20)
never = threshold_rule(graph, rule_id='tutorial_never', threshold=100)
policy_space = SemanticAddressSpace(
    (
        manage_rule_occurrence(common, 'common'),
        manage_rule_occurrence(eligible, 'eligible'),
        manage_rule_occurrence(never, 'never_left'),
        manage_rule_occurrence(never, 'never_right'),
    )
)
topology_policy = Policy(
    'tutorial_policy_topology',
    PolicyAll(
        (
            PolicyOccurrence('common'),
            PolicyAny(
                (
                    PolicyOccurrence('eligible'),
                    PolicyAll(
                        (
                            PolicyOccurrence('never_left'),
                            PolicyOccurrence('never_right'),
                            PolicyUnify(
                                address('never_left', 'person'),
                                address('never_right', 'person'),
                            ),
                            PolicyCompare.gt(
                                address('never_left', 'age'),
                                address('never_right', 'age'),
                            ),
                        )
                    ),
                )
            ),
        )
    ),
    version='1',
)
topology_run = (
    graph.query(topology_policy, address_space=policy_space)
    .bind(address('common', 'person'), alice)
    .select('age', address('common', 'age'))
    .plan()
    .run()
)
assert isinstance(topology_run, GoalPlanRunV1)
topology_row = topology_run.run.effective.canonical_result.rows[0]
assert topology_row.anchor is not None
explained = topology_run.explain(
    ExplainTargetV1('effective', 'row', topology_row.anchor.anchor_digest)
)
assert explained.engine_evidence == 'native_detached_recomputed'
assert explained.proof_parity == 'not_claimed'
assert explained.evidence_graph is not None
assert explained.policy_projection is not None

projection = explained.policy_projection.evaluation
node_kinds = {node.kind for node in projection.nodes}
branch_states = {
    state.state
    for node in projection.nodes
    for state in node.branch_states
}
assert node_kinds == {'all', 'any', 'occurrence', 'unify', 'compare'}
assert {'holds', 'fails', 'not_reached'} <= branch_states

{
    'logical_conclusion': explained.logical_conclusion,
    'evidence_paths': [(path.status, len(path.rules)) for path in explained.evidence_graph.paths],
    'policy_root_state': projection.root_state,
    'policy_node_kinds': sorted(node_kinds),
    'branch_states': sorted(branch_states),
}


## 6. Portable deterministic profile: Native + Soufflé + ProbLog

The V1 portable profile is implemented, but it is intentionally narrow. FactGraph materializes one finite, positive, deterministic effective relation into an isolated Store and runs each engine against that same relation. It compares **canonical selected-row sets** only.

The profile does not claim identical confidence, support/proof trees, engine configuration behavior, general negation, aggregate/recursive semantics, arbitrary external functions, or a fallback to Native when another adapter is unavailable. Those outcomes remain explicit per-engine frames.

In [ ]:
portable = (
    graph.query(age_only_rule(graph))
    .bind(address('target', 'person'), alice)
    .select('age', address('target', 'age'))
    .plan(profile=portable_deterministic_profile_v1())
    .run()
)
assert isinstance(portable, GoalPlanRunV1)
frames = [(frame.engine, frame.status) for frame in portable.run.effective.engine_results]
assert frames == [('native', 'succeeded'), ('souffle', 'succeeded'), ('problog', 'succeeded')]
assert portable.run.effective.assessment.parity == 'equivalent'
assert portable.replay().status == 'matched'

{'engine_frames': frames, 'selected_row_parity': portable.run.effective.assessment.parity}


### Cross-entity field navigation and comparison

This is a deliberately harder portable case: the Policy contains two different occurrences of the same Rule, navigates each `person` endpoint to `age`, and compares the results. It proves that an `Any`/`All`-style Policy topology is not flattened into a single Rule string, and that Native, Soufflé, and ProbLog consume the same sealed relation without a Native fallback.

As always, `equivalent` means canonical selected-row-set parity only—not equal proof trees, certainty, or provenance across engines.

In [ ]:
cross_values = person_values_rule(graph)
cross_space = SemanticAddressSpace(
    (
        manage_rule_occurrence(cross_values, 'older'),
        manage_rule_occurrence(cross_values, 'younger'),
    )
)
older_person = address('older', 'person')
younger_person = address('younger', 'person')
cross_policy = Policy(
    'tutorial_cross_entity_age_order',
    PolicyAll(
        (
            PolicyOccurrence('older'),
            PolicyOccurrence('younger'),
            PolicyCompare.gt(
                PolicyFieldNavigation(older_person, FieldPath('Person', 'age')),
                PolicyFieldNavigation(younger_person, FieldPath('Person', 'age')),
            ),
        )
    ),
    version='1',
)
cross_portable = (
    graph.query(cross_policy, address_space=cross_space)
    .select('older', older_person)
    .select('younger', younger_person)
    .select('older_age', address('older', 'age'))
    .select('younger_age', address('younger', 'age'))
    .plan(profile=portable_deterministic_profile_v1())
    .run()
)
cross_frames = [(frame.engine, frame.status) for frame in cross_portable.run.effective.engine_results]
assert cross_frames == [('native', 'succeeded'), ('souffle', 'succeeded'), ('problog', 'succeeded')]
assert cross_portable.run.effective.assessment.parity == 'equivalent'
cross_row = dict(cross_portable.run.effective.canonical_result.rows[0].values)
assert cross_row['older'].value != cross_row['younger'].value
assert (cross_row['older_age'].value, cross_row['younger_age'].value) == (22, 19)

{'engine_frames': cross_frames, 'selected_row_parity': cross_portable.run.effective.assessment.parity}


## 7. Relation providers compose with Query; they are not bare Rules

A `RelationProviderV1` is a constrained pre-engine input. It has neither an invented output head nor a standalone projection, so `graph.query(provider)` intentionally rejects. Instead, `.using(provider)` attaches it to an existing Rule/Policy Query; it materializes one finite typed relation and a sealed receipt before the engines execute. Replay uses the captured relation and receipt, never calls the provider again.

The callback is trusted same-process code. FactGraph detects a persistent source-view mutation and refuses to seal a result, but it is not a sandbox and cannot roll a malicious mutation back.

In [ ]:
provider_calls: list[str] = []


def materialize_age_lookup(request):
    provider_calls.append(request.request_digest)
    return ProviderMaterializationV1(
        request.provider_digest,
        request.request_digest,
        'tutorial:age-lookup',
        sha256_token(b'tutorial:age-lookup:receipt'),
        request.supplied_predicate_ids,
        (
            ProviderRelationRowV1(
                age_predicate_id,
                (GoalValueV1('entity_ref', alice_ref), GoalValueV1('int', 30)),
                'tutorial:age-lookup-row',
            ),
        ),
    )


age_lookup = RelationProviderV1(
    'tutorial.age_lookup',
    '1',
    'lookup',
    sha256_token(b'tutorial:age-lookup:code'),
    (provider_binding_slot_v1('target', 'person'),),
    (age_predicate_id,),
    materialize_age_lookup,
)
provider_run = (
    graph.query(age_only_rule(graph))
    .bind(address('target', 'person'), alice)
    .select('age', address('target', 'age'))
    .using(age_lookup)
    .plan()
    .run()
)
assert isinstance(provider_run, GoalPlanRunV1)
assert dict(provider_run.run.effective.canonical_result.rows[0].values)['age'].value == 30
assert len(provider_calls) == 1
assert len(provider_run.run.replay_payload.provider_receipts) == 1
assert provider_run.replay().status == 'matched'
assert len(provider_calls) == 1

{'age_from_provider': 30, 'provider_calls': len(provider_calls), 'receipt_count': len(provider_run.run.replay_payload.provider_receipts)}


## 8. Immutable candidate comparison and replay

A candidate is a separately compiled Rule/Policy target evaluated against the **same sealed effective world**. It is not an AST patch, mutable policy overlay, publication request, or causal claim. The captured world may contain the union of dependencies required by both sides, while each isolated evaluator receives only its declared subset.

In [ ]:
comparison_run = (
    graph.query(age_only_rule(graph))
    .bind(address('target', 'person'), alice)
    .select('age', address('target', 'age'))
    .plan(candidate=age_with_score_guard_rule(graph))
    .run()
)
assert isinstance(comparison_run, GoalPlanRunV1)
assert comparison_run.comparison is not None
assert comparison_run.comparison.result_relation == 'equivalent'
assert comparison_run.comparison.causal_attribution == 'not_claimed'
assert comparison_run.replay().status == 'matched'

{
    'compiled_body_equal': comparison_run.comparison.compiled_body_equal,
    'authored_structure_relation': comparison_run.comparison.authored_structure_relation,
    'result_relation': comparison_run.comparison.result_relation,
    'causal_attribution': comparison_run.comparison.causal_attribution,
}


## 9. What this closure does and does not promise

**Implemented in FactGraph V1**

- direct Rule/Policy Query targets with structured ports, branch-total select, one-hop field navigation, result modes, expectations, providers, Scenario worlds, candidate comparison, replay, and explicit Explain;
- the finite Scenario algebra (set/exact-set/member/relation/entity/assertion removal and grounded ephemeral entities), with deterministic conflict resolution and no ledger persistence;
- real Native/Soufflé/ProbLog canonical selected-row parity for `portable_deterministic_v1`; and
- captured-world replay and a lazy, positive-row native EvidenceGraph plus authored Policy overlay.

**Explicitly not promised**

- automatic product approval, Agent permissions, Package lookup, source authority, Action execution, or translation from raw Agent prose (those belong to Meander);
- global closed-world negation, generic why-not proof for a zero result, a negative EvidenceGraph, mutable Rule/Policy patches, or scenario persistence;
- universal cross-engine language/proof/certainty equivalence, engine fallback, arbitrary engine-time callbacks, or provider sandboxing.

These boundaries are why the interface is unified at the Query/Run/composition/audit layer without falsely declaring Rules, external operators, and Actions semantically identical.

## Next steps

For Meander integration, compile an authorized Agent plan into this already-pinned FactGraph Query/GoalPlan contract. Keep Agent translation, Package/Policy selection, source authority, retention, UI rendering, and Action approval outside this notebook's execution surface.